# Individual Assignment
## Code and Model Pipeline

### Adam Hrehovcik

## Notebook Introduction

This notebook implements a small, end-to-end text modeling pipeline on IMDB reviews. It first builds a word-level trigram language model trained on the full training vocabulary, then train an LSTM next-token model under a restricted vocabulary budget (keeping only the most frequent tokens). Finally, it compare likelihood-based metrics and generation behavior, and train a simple discriminator to distinguish real sentences from RNN-generated sentences.


## Setup

#### Importing the data
First the training data are loaded into a single list by iterating over the file paths in both `pos` and `neg` folders in train/test splits.  
Although labels are available, this part of the assignment uses the text corpus for language modeling rather than sentiment prediction. Thus, the sentiment labels are ignored.

In [2]:
from pathlib import Path

def iter_over_files(root, split):
    root = Path(root) / split
    for label_directory, label in [("pos", 1), ("neg", 0)]:
        for file_path in (root/label_directory).iterdir():
            text = file_path.read_text(encoding = "utf-8", errors = "ignore")
            yield text, label

train_data = list(iter_over_files("aclImdb", "train"))
test_data = list(iter_over_files("aclImdb", "test"))


In [3]:
for i, (text, y) in zip(range(3), train_data):
    print(y, text[:120])

1 Bromwell High is a cartoon comedy. It ran at the same time as some other programs about school life, such as "Teachers".
1 Homelessness (or Houselessness as George Carlin stated) has been an issue for years but never a plan to help those on th
1 Brilliant over-acting by Lesley Ann Warren. Best dramatic hobo lady I have ever seen, and love scenes in clothes warehou


## Part 1: Tokenization

#### Building a tokenizer

Here, the raw review text is converted into sentence-level token sequences and add boundary tokens (`<START>`, `<END>`). Tokenization is word-based, not character/subword based.
- normalize text (lowercase, strip whitespace) and remove HTML break tags
- split reviews into sentences using simple punctuation rules
- clean each sentence to `[a-z0-9 ]`, split on whitespace, and wrap each sentence with `<START>` and `<END>`
- output format is a list of tokenized sentences per review (keeps training/evaluation sentence-based)

The tokenization intentionally simple so later differences come from the models (trigram vs RNN), not from heavy preprocessing.

In [4]:
import re
def tokenize_text(text):
    tokenized_sentences = []
    text = text.lower().strip()
    text = re.sub(r"<br\s*/?>", " ", text)
    sentences = re.split(r"[.!?]+", text)
    for sentence in sentences:
        sentence = re.sub(r"[^a-z0-9\s]", "", sentence)
        sentence = re.sub(r"<br\s*/?>", " ", sentence)
        tokens = sentence.split()
        if tokens:
            tokenized_sentences.append(["<START>"] + tokens + ["<END>"])
    return tokenized_sentences

#### Building the vocabulary and encoding

The following code:
- prepares the function to build a full vocabulary from data, reserving special tokens (`<PAD>`, `<UNK>`, `<START>`, `<END>`)
- creates `token_to_id` mapping and encode each tokenized sentence to integer IDs (unknown tokens map to `<UNK>`)


In [5]:
def build_vocabulary(dataset):
    special_tokens = ["<PAD>", "<UNK>", "<START>", "<END>"]
    vocab = set() 
    for (text, y) in dataset: 
        tokenized_sentences = tokenize_text(text) 
        for sentence in tokenized_sentences:
            for token in sentence: 
                vocab.add(token)

    vocab = sorted(vocab - set(special_tokens)) 
    vocab = special_tokens + vocab 
                
    token_to_id = {token: i for i, token in enumerate(vocab)} 
    return vocab, token_to_id

def encode_text(text, token_to_id):
    unknown = token_to_id["<UNK>"]
    encoded_text = []
    tokenized_text = tokenize_text(text)
    for sentence in tokenized_text:
        encoded_text.append([token_to_id.get(token, unknown) for token in sentence])
    return encoded_text

#### Implementing the tokenizer

The code below:

- builds the full full vocabulary by impelenting function from above on the full set of training data
- encodes train/test reviews to produce integer sequences for later language modelling tasks

In [6]:
full_vocab, full_token_to_id = build_vocabulary(train_data)
id_to_token_full = {idx: token for token, idx in full_token_to_id.items()}

X_train_full = [encode_text(text, full_token_to_id) for text, y in train_data]
y_train = [y for text, y in train_data]

X_test_full = [encode_text(text, full_token_to_id) for text, y in test_data]
y_test = [y for text, y in test_data]


vocab = full_vocab
token_to_id = full_token_to_id


vocab_size = len(vocab)
print(f"Vocabulary length: {vocab_size}")

sample_text = train_data[0][0]

print("\nOriginal review snippet:")
print(sample_text[:120])

print("\nTokenized first sentence:")
print(tokenize_text(sample_text)[0])

print("\nEncoded first sentence:")
print(X_train_full[0][0])


Vocabulary length: 104100

Original review snippet:
Bromwell High is a cartoon comedy. It ran at the same time as some other programs about school life, such as "Teachers".

Tokenized first sentence:
['<START>', 'bromwell', 'high', 'is', 'a', 'cartoon', 'comedy', '<END>']

Encoded first sentence:
[2, 13328, 42838, 47738, 2110, 15504, 19117, 3]


## Part 2: Non-deep learning model (N-gram)

#### Building the trigram

This block constructs trigram context counts from the tokenized training data by:

- tokenizing each training review again and aggregating all sentences
- adding an extra `<START>` at the beginning of each sentence to support the initial trigram context (`<START>, <START> -> first word`)
- converting each sentence into overlapping trigrams and store counts in a nested dict:
  - key: `(w1, w2)` context
  - value: counts over possible `w3`

The output is a compact frequency table that can be turned into conditional probabilities for generation and evaluation.


In [7]:
def build_trigram_counts(dataset):
    all_sentences = []
    for text, y in dataset:
        tokenized_text = tokenize_text(text)
        for sentence in tokenized_text:
            sentence = ["<START>"] + sentence
            all_sentences.append(sentence)

    trigrams = []
    for sentence in all_sentences:
        for i in range(2,len(sentence)):
            tri_token = (sentence[i-2], sentence[i-1], sentence[i])
            trigrams.append(tri_token)

    trigram_counts = {}
    for w1, w2, w3 in trigrams:
        context = (w1, w2)
        if context not in trigram_counts:
            trigram_counts[context] = {}
        trigram_counts[context][w3] = trigram_counts[context].get(w3, 0) + 1

    return trigram_counts

#### Implementing the smoothing (Trigram)

Below, the code defines a smoothed trigram probability function (using the Laplace smoothing) so unseen trigrams still get non-zero probability.

In [8]:
trigram_counts = build_trigram_counts(train_data)

def trigram_prob(w1, w2, w3, vocab_size, k = 1):
    context = (w1,w2)
    count = trigram_counts.get(context, {}).get(w3,0)
    context_total = sum(trigram_counts.get(context, {}).values())
    prob = (count + k) / (context_total + k * vocab_size)

    return prob

#### Text generation (Trigram)

This section samples text from the trigram model starting from a fixed start context.

It supports two decoding modes:
  - `greedy`: pick argmax token
  - `top-10`: sample from the top 10 tokens by probability

The code stops early if `<END>` is produced to simulate end of sentence.

In [12]:
import random

def generate_text(trigram_counts, vocab, vocab_size, max_length = 50, method = "greedy", k = 1):
    w1, w2 = "<START>", "<START>"
    generated_text = []
    for i in range(max_length):
        candidates = list(trigram_counts.get((w1,w2), {}).keys())
        if not candidates:
            candidates = [w for w in vocab if w != "<START>"]

        probabilities = {}

        for candidate in candidates:
            prob = trigram_prob(w1, w2, candidate, vocab_size, k)
            probabilities[candidate] = prob

        sampled_w3 = None

        if method == "greedy":
            best_prob = -1
            for w3, prob in probabilities.items():
                if prob > best_prob:
                    best_prob = prob
                    sampled_w3 = w3

        elif method == "top-10":
            candidate_pairs = list(probabilities.items())
            candidate_pairs.sort(key = lambda x: x[1], reverse = True)
            top_pairs = candidate_pairs[:10]
            total = sum(p for w, p in top_pairs)
            top_candidates = []
            top_probs = []
            for w, p in top_pairs:
                top_candidates.append(w)
                top_probs.append(p / total)
            r = random.random()
            cum = 0
            for w, p in zip(top_candidates, top_probs):
                cum += p
                if r <= cum:
                    sampled_w3 = w
                    break
        else:
            raise ValueError("method must be 'greedy' or 'top-10'")
        if sampled_w3 == "<END>":
            break
        generated_text.append(sampled_w3)
        w1, w2 = w2, sampled_w3
    return " ".join(generated_text)

print("Greedy Sampling:")
print(generate_text(trigram_counts, vocab, vocab_size, max_length=80, method="greedy"))
print("\n")
for i in range(5):
    print(f"Sample {i+1}:")
    print(generate_text(trigram_counts, vocab, vocab_size, max_length=80, method="top-10"))

Greedy Sampling:
the film is a very good


Sample 1:
this film was released
Sample 2:
the movie is about two hours the audience
Sample 3:
this is not a good movie
Sample 4:
the only way they have a very nice job considering hers is a very good film
Sample 5:
in addition the great movie


## Part 3: Deep Learning Model (RNN)

#### Reducing training vocabulary (RNN)

The section below creates a second vocabulary containing only the most frequent training tokens (plus special tokens) and re-encodes data.
This sets up the RNN as a constrained model where rare words collapse to `<UNK>`. This is because the full vocabulary is too large and sparse for a small RNN to learn effectively within this assignment’s compute/data budget, so restricting to the top 25% concentrates learning on frequent patterns while mapping rare words to <UNK>.

So, the code does:
- count token frequencies on the training split and keep only the top 20% most frequent tokens (plus special tokens)
- re-encode train/test using this reduced mapping so all out-of-vocabulary tokens become `<UNK>`
- keep full-vocab encodings separate so trigram evaluation remains full-vocab

In [10]:
def build_reduced_vocabulary(dataset, top_fraction=0.2):
    special_tokens = ["<PAD>", "<UNK>", "<START>", "<END>"]

    freq = {}

    for text, y in dataset:
        tokenized_sentences = tokenize_text(text)
        for sentence in tokenized_sentences:
            for token in sentence:
                if token in special_tokens:
                    continue
                freq[token] = freq.get(token, 0) + 1

    sorted_tokens = sorted(freq.items(), key=lambda x: x[1], reverse=True)
    top_n = max(1, int(len(sorted_tokens) * top_fraction))

    kept_tokens = [token for token, count in sorted_tokens[:top_n]]

    vocab = special_tokens + kept_tokens
    token_to_id = {token: i for i, token in enumerate(vocab)}

    return vocab, token_to_id, top_n, len(sorted_tokens)

reduced_vocab, reduced_token_to_id, reduced_top_n, reduced_base_size = build_reduced_vocabulary(
    train_data,
    top_fraction=0.2
)

X_train_reduced = [encode_text(text, reduced_token_to_id) for text, y in train_data]
y_train = [y for text, y in train_data]

X_test_reduced = [encode_text(text, reduced_token_to_id) for text, y in test_data]
y_test = [y for text, y in test_data]

# keep reduced set as active data for the DL section
X_train = X_train_reduced
X_test = X_test_reduced

reduced_vocab_size = len(reduced_vocab)
print(f"Reduced vocab size: {reduced_vocab_size} (top {reduced_top_n} of {reduced_base_size} non-special tokens)")


Reduced vocab size: 20823 (top 20819 of 104096 non-special tokens)


#### Training data preparation (RNN)

The following code does:

- flatten the encoded reviews into a single list of encoded sentences for train and test.
- build supervised next-token pairs:
  - for each sentence position `i`, context is `sentence[:i]` and target is `sentence[i]`
  - left-pad/truncate contexts to a fixed `max_len`
  - optionally subsample positions using `stride` to reduce redundancy and dataset size
- shuffle and split training pairs into train/validation
- print the final number of training/validation/test pairs and proportion of <UNK> tokens in the training data

The idea is to cap context length and subsample positions to control dataset size and keep later model training efficient.

In [11]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

train_sentences_reduced = []
for review in X_train_reduced:
    for sent in review:
        train_sentences_reduced.append(sent)

test_sentences_reduced = []
for review in X_test_reduced:
    for sent in review:
        test_sentences_reduced.append(sent)

# check to see the sentence length stats, to ensure appropriate maximum length of context the model will learn from
# (given that training is done on per sentence basis)
lengths = [len(s) for s in train_sentences_reduced]
print("Num train sentences:", len(train_sentences_reduced))
print("Median len:", int(np.median(lengths)))

pad_id = reduced_token_to_id["<PAD>"]

def padding(sequence, max_len):
    sequence = sequence[-max_len:]
    return [pad_id] * (max_len - len(sequence)) + sequence

last_k = 5

def build_dl_training_data(encoded_sentences, max_len = 20, stride = 5):
    X = []
    y = []
    for sentence in encoded_sentences:
        if len(sentence) < 3:
            continue

        for i in range(1, len(sentence)):
            target = sentence[i]

            if i not in (1, 2) and (i % stride) != 0:
                continue

            context = sentence[:i]
            X.append(padding(context, max_len))
            y.append(target)
    X = np.array(X, dtype = np.int32)
    y = np.array(y, dtype = np.int32)
    return X, y

max_len = 20

X_dl_train, y_dl_train = build_dl_training_data(train_sentences_reduced, max_len = max_len, stride = 5)
X_dl_test, y_dl_test = build_dl_training_data(test_sentences_reduced, max_len = max_len, stride = 5)


idx = np.arange(len(X_dl_train))
np.random.shuffle(idx)
X_dl_train, y_dl_train = X_dl_train[idx], y_dl_train[idx]

validation_frac = 0.1
val_n = int(len(X_dl_train) * validation_frac)

X_tr, y_tr = X_dl_train[val_n:], y_dl_train[val_n:]
X_val, y_val = X_dl_train[:val_n], y_dl_train[:val_n]

print("Training pairs:", X_tr.shape, y_tr.shape)
print("Validation pairs:", X_val.shape, y_val.shape)
print("Test pairs:", X_dl_test.shape, y_dl_test.shape)

unk_id = reduced_token_to_id["<UNK>"]

total_tokens = 0
unk_count = 0

for sent in train_sentences_reduced:
    total_tokens += len(sent)
    unk_count += sum(1 for token in sent if token == unk_id)

unk_prop = unk_count / total_tokens

print(f"Total tokens: {total_tokens}")
print(f"<UNK> count: {unk_count}")
print(f"<UNK> proportion: {unk_prop:.4f} ({unk_prop*100:.2f}%)")

Num train sentences: 332395
Median len: 17
Training pairs: (1577682, 20) (1577682,)
Validation pairs: (175298, 20) (175298,)
Test pairs: (1717888, 20) (1717888,)
Total tokens: 6440903
<UNK> count: 174056
<UNK> proportion: 0.0270 (2.70%)


#### Model training (RNN)

The code defines a compact next-token model:
  - `Embedding(..., mask_zero=True)` to ignore padding
  - `LSTM(hidden_units)` to summarize context
  - `Dense(vocab_size, softmax)` to predict next token ID
- Train with cross-entropy, validate each epoch, and early-stop on `val_loss`.
- Evaluate on held-out test pairs and report loss and perplexity

In [50]:
embedding_dim = 128
hidden_units = 128

model = keras.Sequential(
    [layers.Input(shape = (max_len, )),
     layers.Embedding(input_dim = reduced_vocab_size, output_dim = embedding_dim, mask_zero = True),
     layers.LSTM(hidden_units),
     layers.Dense(reduced_vocab_size, activation = "softmax")
     ])

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["sparse_categorical_accuracy"])

model.summary()

callbacks = [keras.callbacks.EarlyStopping(monitor="val_loss", patience=1, restore_best_weights=True)]

history = model.fit(
    X_tr, y_tr,
    validation_data = (X_val, y_val),
    epochs = 5,
    batch_size = 128,
    callbacks = callbacks)

subset_n = min(200000, len(X_dl_test))
subset_idx = np.random.choice(len(X_dl_test), size = subset_n, replace=False)
X_test_subset = X_dl_test[subset_idx]
y_test_subset = y_dl_test[subset_idx]

val_loss, val_acc = model.evaluate(X_val, y_val, verbose=0)
test_loss, test_acc = model.evaluate(X_dl_test, y_dl_test, verbose=0)

print("Val loss:", val_loss, "Val perplexity:", float(np.exp(val_loss)))
print("Test loss:", test_loss, "Test perplexity:", float(np.exp(test_loss)))


Model: "sequential_6"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_6 (Embedding)         │ (None, 20, 128)        │     2,665,344 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_6 (LSTM)                   │ (None, 128)            │       131,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 20823)          │     2,686,167 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 5,483,095 (20.92 MB)

 Trainable params: 5,483,095 (20.92 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/5
12326/12326 ━━━━━━━━━━━━━━━━━━━━ 711s 58ms/step - loss: 5.7633 - sparse_categorical_accuracy: 0.1350 - val_loss: 5.4488 - val_sparse_categorical_accuracy: 0.1586
Epoch 2/5
12326/12326 ━━━━━━━━━━━━━━━━━━━━ 770s 62ms/step - loss: 5.2882 - sparse_categorical_accuracy: 0.1658 - val_loss: 5.3093 - val_sparse_categorical_accuracy: 0.1680
Epoch 3/5
12326/12326 ━━━━━━━━━━━━━━━━━━━━ 843s 68ms/step - loss: 5.0953 - sparse_categorical_accuracy: 0.1777 - val_loss: 5.2634 - val_sparse_categorical_accuracy: 0.1726
Epoch 4/5
12326/12326 ━━━━━━━━━━━━━━━━━━━━ 741s 59ms/step - loss: 4.9583 - sparse_categorical_accuracy: 0.1864 - val_loss: 5.2559 - val_sparse_categorical_accuracy: 0.1745
Epoch 5/5
12326/12326 ━━━━━━━━━━━━━━━━━━━━ 741s 60ms/step - loss: 4.8511 - sparse_categorical_accuracy: 0.1940 - val_loss: 5.2738 - val_sparse_categorical_accuracy: 0.1738
Val loss: 5.255904674530029 Val perplexity: 191.6948288622625
Test loss: 5.226425647735596 Test perplexity: 186.1263320514026


In [51]:
model.save('my_final_model.keras')

#### Text generation

The section below generates tokens starting from `<START>`:
  - pads the running context to `max_len`
  - predicts next-token probabilities
  - decodes via greedy or top-k sampling
  - appends predicted token and continue until `<END>` or max tokens
At the end, it decodes IDs back to tokens using the reduced vocabulary mapping

In [81]:
id_to_token_reduced = {idx: token for token, idx in reduced_token_to_id.items()}

def sample_from_top_k(probs, k = 10):
    top_idx = np.argpartition(probs, -k)[-k:]
    top_probs = probs[top_idx]
    top_probs = top_probs / top_probs.sum()
    choice = np.random.choice(top_idx, p=top_probs)
    return int(choice)

def generate_rnn(model, token_to_id, id_to_token, max_len = 20, max_tokens = 50, method = "greedy", k = 10):
    start_id = token_to_id["<START>"]
    end_id = token_to_id["<END>"]

    context = [start_id]
    generated_text = []

    for _ in range(max_tokens):
        x = np.array([padding(context, max_len)])
        probs = model.predict(x, verbose = 0)[0]

        if method == "greedy":
            next_id = int(np.argmax(probs))

        elif method == "top-k":
            next_id = sample_from_top_k(probs, k = k)

        else:
            raise ValueError("method must be 'greedy' or 'top-k'")

        if next_id == end_id:
            break

        generated_text.append(id_to_token.get(next_id, "<UNK>"))
        context.append(next_id)

    return " ".join(generated_text)

print("\nRNN Greedy:")
print(generate_rnn(model, reduced_token_to_id, id_to_token_reduced, max_len=max_len, max_tokens=50, method="greedy"))

print("\nRNN Top-k (k=10):")
for i in range(5):
    print(f"Sample {i+1}: {generate_rnn(model, reduced_token_to_id, id_to_token_reduced, max_len=max_len, max_tokens=50, method='top-k', k=10)}")


RNN Greedy:
the film is a very good film

RNN Top-k (k=10):
Sample 1: its a great movie but its not a great movie for the time of this film
Sample 2: the film was a little too long
Sample 3: its a good movie that i was not even surprised with it
Sample 4: and this isnt a bad film
Sample 5: i cant even consider how this film is about


#### Trigram vs. RNN comparison

The section below compares trigram and RNN language models using perplexity:

- flattens training and test sets into sentence-level encoded sequences
- builds trigram counts from encoded training sentences
- computes trigram loss and perplexity over test sentences using smoothed probabilities
- repeats the trigram evaluation for both full and reduced vocabularies
- evaluates the trained RNN on a held-out reduced test subset
- converts RNN cross-entropy loss into perplexity
- prints all losses and perplexities side-by-side for direct comparison

In [55]:
full_test_sentences = []
for review in X_test_full:
    for sent in review:
        full_test_sentences.append(sent)

full_train_sentences = []
for review in X_train_full:
    for sent in review:
        full_train_sentences.append(sent)


def evaluate_trigram_perplexity(encoded_sentences, id_to_token_map, trigram_vocab_size, k = 1, subset_size = None):
    total_log_prob = 0.0
    total_tokens = 0

    for sentence in encoded_sentences:
        if len(sentence) < 3:
            continue

        decoded = [id_to_token_map[idx] for idx in sentence]
        tokens = ["<START>"] + decoded

        for i in range(2, len(tokens)):
            w1 = tokens[i-2]
            w2 = tokens[i-1]
            w3 = tokens[i]

            prob = trigram_prob(w1, w2, w3, trigram_vocab_size, k)
            prob = max(prob, 1e-10)
            total_log_prob += -np.log(prob)
            total_tokens += 1

        if subset_size is not None and total_tokens >= subset_size:
            break

    avg_loss = total_log_prob / total_tokens
    perplexity = np.exp(avg_loss)

    return avg_loss, perplexity

def build_trigram_counts_from_encoded(encoded_sentences, id_to_token):
    trigram_counts_local = {}
    for sent in encoded_sentences:
        if len(sent) < 3:
            continue
        tokens = ["<START>"] + [id_to_token[i] for i in sent]
        for j in range(2, len(tokens)):
            ctx = (tokens[j-2], tokens[j-1])
            w3 = tokens[j]
            if ctx not in trigram_counts_local:
                trigram_counts_local[ctx] = {}
            trigram_counts_local[ctx][w3] = trigram_counts_local[ctx].get(w3, 0) + 1
    return trigram_counts_local

trigram_counts = build_trigram_counts_from_encoded(full_train_sentences, id_to_token_full)

trigram_loss, trigram_perplexity = evaluate_trigram_perplexity(
    full_test_sentences,
    id_to_token_full,
    vocab_size,
    k=1,
    subset_size=200000
)

trigram_counts = build_trigram_counts_from_encoded(train_sentences_reduced, id_to_token_reduced)

trigram_loss_reduced, trigram_perplexity_reduced = evaluate_trigram_perplexity(
    test_sentences_reduced,
    id_to_token_reduced,
    reduced_vocab_size,
    k=1,
    subset_size=200000
)

rnn_test_loss, rnn_test_acc = model.evaluate(X_test_subset, y_test_subset, verbose=0)
rnn_perplexity = float(np.exp(rnn_test_loss))

print("\nModel Comparison")
print("=" * 60)
print(f"{'Model':<30}{'Loss':>12}{'Perplexity':>15}")
print("-" * 60)

print(f"{'Trigram (full vocab)':<30}{trigram_loss:>12.4f}{trigram_perplexity:>15.2f}")
print(f"{'Trigram (reduced vocab)':<30}{trigram_loss_reduced:>12.4f}{trigram_perplexity_reduced:>15.2f}")
print(f"{'RNN (reduced vocab)':<30}{rnn_test_loss:>12.4f}{rnn_perplexity:>15.2f}")

print("=" * 60)



Model Comparison
Model                                 Loss     Perplexity
------------------------------------------------------------
Trigram (full vocab)                9.9943       21901.90
Trigram (reduced vocab)             8.3235        4119.68
RNN (reduced vocab)                 5.2159         184.18


## Part 4: Deep Learning Classifier

#### Preprocessing real sentences

The section below prepares real samples for the discriminator:
- inspects sentence length distribution to guide input length choice
- defines a pad or truncate function to fix sentence length
- randomly samples a subset of reduced training sentences
- pads them to max_len_disc and converts to NumPy arrays
- assigns label 1 to indicate real data


In [58]:
start_id = reduced_token_to_id["<START>"]
end_id = reduced_token_to_id["<END>"]

lengths = [len(s) for s in train_sentences_reduced]

print("90th percentile:", np.percentile(lengths, 90))
print("95th percentile:", np.percentile(lengths, 95))

max_len_disc = 30

def pad_or_truncate(sequence, max_length):
    sequence = sequence[:max_length]
    output = sequence + [pad_id] * (max_length - len(sequence))
    return output

n_real = 2000
idx = np.random.choice(len(train_sentences_reduced), size=n_real, replace=False) 
real_sents = [train_sentences_reduced[i] for i in idx] 
X_real = np.array([pad_or_truncate(sent, max_len_disc) for sent in real_sents], dtype=np.int32) 
y_real = np.ones((n_real,), dtype=np.int32)

90th percentile: 34.0
95th percentile: 41.0


#### Generating fake sentences

The code below generated fake data to be fed into the discriminator:
- defines generate_rnn_ids to produce sentences as token ID sequences using the trained RNN with top-k sampling
- generates n_fake synthetic sentences
- pads or truncates them to the discriminator input length
- assigns label 0 to indicate fake data

In [60]:
def generate_rnn_ids(model, token_to_id, max_len = 20, max_tokens = 40, method = "top-k", k = 10):
    context = [token_to_id["<START>"]]
    output = [token_to_id["<START>"]]

    for i in range(max_tokens):
        x = np.array([padding(context, max_len)])
        probs = model(x, training=False).numpy()[0]

        if method == "greedy":
            next_id = int(np.argmax(probs))

        elif method == "top-k":
            next_id = sample_from_top_k(probs, k = k)

        output.append(next_id)
        context.append(next_id)

        if next_id == token_to_id["<END>"]:
            break

    return output

n_fake = n_real
fake_sents = [generate_rnn_ids(model, reduced_token_to_id, max_len = 20, max_tokens = 40, method = "top-k", k = 10) for i in range(n_fake)]

X_fake = np.array([pad_or_truncate(sent, max_len_disc) for sent in fake_sents], dtype=np.int32)
y_fake = np.zeros((n_fake,), dtype=np.int32)

#### Combining and splitting the data

This segment concatenates real and fake sentences into `X, y`, shuffles the data, and then splits into validation and training sets.

In [61]:
X = np.concatenate([X_real, X_fake], axis = 0)
y = np.concatenate([y_real, y_fake], axis = 0)

perm = np.random.permutation(len(y))
X, y = X[perm], y[perm]

val_frac = 0.2
n_val = int(len(y) * val_frac)

X_valid, y_valid = X[:n_val], y[:n_val]
X_tra, y_tra = X[n_val:], y[n_val:]

#### Discriminative Model

The section below defines and trains the discriminator model:

- builds a neural network with an embedding layer, Gaussian noise for regularisation, global average pooling, a dense ReLU layer with dropout, and a final sigmoid output for binary classification
- compiles the model using Adam optimisation, binary cross-entropy loss, and accuracy plus AUC as evaluation metrics
- sets up early stopping based on validation loss to prevent overfitting
- trains the model on the real and fake training data with validation monitoring

In [ ]:
disc = keras.Sequential([
    layers.Input(shape = (max_len_disc, )),
    layers.Embedding(input_dim = reduced_vocab_size, output_dim = 64, mask_zero = True),
    layers.GaussianNoise(0.05),
    layers.GlobalAveragePooling1D(),
    layers.Dense(256, activation = "relu"),
    layers.Dropout(0.3),
    layers.Dense(1, activation = "sigmoid")
])

disc.compile(
    optimizer = keras.optimizers.Adam(1e-3),
    loss = "binary_crossentropy",
    metrics = ["accuracy", keras.metrics.AUC(name = "AUC")]
)

disc.summary()

early_stop = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=4,
    min_delta=1e-4,
    restore_best_weights=True
)


hist = disc.fit(
    X_tra, y_tra,
    validation_data = (X_valid, y_valid),
    epochs = 30,
    batch_size = 128
)

Model: "sequential_17"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_17 (Embedding)        │ (None, 30, 64)         │     1,332,672 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gaussian_noise_1                │ (None, 30, 64)         │             0 │
│ (GaussianNoise)                 │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d_10     │ (None, 64)             │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_27 (Dense)                │ (None, 256)            │        16,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_10 (Dropout)            │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_28 (Dense)                │ (None, 1)              │           257 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,349,569 (5.15 MB)

 Trainable params: 1,349,569 (5.15 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - AUC: 0.8060 - accuracy: 0.7306 - loss: 0.6694 - val_AUC: 0.9502 - val_accuracy: 0.7650 - val_loss: 0.6104
Epoch 2/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - AUC: 0.9492 - accuracy: 0.8456 - loss: 0.5076 - val_AUC: 0.9557 - val_accuracy: 0.8875 - val_loss: 0.3861
Epoch 3/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - AUC: 0.9656 - accuracy: 0.9078 - loss: 0.2942 - val_AUC: 0.9630 - val_accuracy: 0.9050 - val_loss: 0.2646
Epoch 4/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - AUC: 0.9776 - accuracy: 0.9284 - loss: 0.2012 - val_AUC: 0.9689 - val_accuracy: 0.9112 - val_loss: 0.2246
Epoch 5/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - AUC: 0.9849 - accuracy: 0.9456 - loss: 0.1588 - val_AUC: 0.9728 - val_accuracy: 0.9150 - val_loss: 0.2087
Epoch 6/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - AUC: 0.9893 - accuracy: 0.9559 - loss: 0.1314 - val_AUC: 0.9753 - val_accuracy: 0.9200 - val_loss: 0.1965
Epoch 7/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step -

In [79]:
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support, roc_auc_score, average_precision_score

probs = disc.predict(X_valid, batch_size=512).ravel()
preds = (probs >= 0.5).astype(int)

cm = confusion_matrix(y_valid, preds)
prec, rec, f1, _ = precision_recall_fscore_support(y_valid, preds, average="binary", zero_division=0)

loss, acc, auc = disc.evaluate(X_valid, y_valid, verbose=0)
roc_auc = roc_auc_score(y_valid, probs)
pr_auc = average_precision_score(y_valid, probs)

print("\n" + "="*50)
print("DISCRIMINATOR EVALUATION (Validation)")
print("="*50)

print(f"\nLoss:                {loss:.4f}")
print(f"Accuracy:            {acc:.4f}")
print(f"ROC AUC:             {roc_auc:.4f}")
print(f"PR AUC:              {pr_auc:.4f}")

print("\nThreshold = 0.5 metrics")
print(f"Precision:           {prec:.4f}")
print(f"Recall:              {rec:.4f}")
print(f"F1 Score:            {f1:.4f}")

print("\nConfusion Matrix:")
print("                 Pred Fake   Pred Real")
print(f"Actual Fake      {cm[0,0]:>10}   {cm[0,1]:>9}")
print(f"Actual Real      {cm[1,0]:>10}   {cm[1,1]:>9}")

2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step



DISCRIMINATOR EVALUATION (Validation)

Loss:                0.3053
Accuracy:            0.9200
ROC AUC:             0.9751
PR AUC:              0.9769

Threshold = 0.5 metrics
Precision:           0.9687
Recall:              0.8651
F1 Score:            0.9140

Confusion Matrix:
                 Pred Fake   Pred Real
Actual Fake             396          11
Actual Real              53         340


## Notebook Conclusion

The trigram model trains extremely quickly, but it achieves substantially worse loss/perplexity and its generations are noticeably less realistic. The RNN takes longer to train, but it reaches much better loss/perplexity and produces slightly more natural-looking text. However, the generator is constrained by the limited compute budget required to train in a reasonable time on a personal laptop. Finally, a very simple discriminator separates real sentences from RNN-generated ones with high performance, suggesting the generator still leaves clear statistical artifacts and the overall generation quality remains limited.
